# Guitar MIDI — pipeline Kaggle

Ce notebook clone une branche d'expérience, monte un dataset privé attaché et lance soit le smoke test, soit le train complet, soit la reconstruction de `data/processed`. Le package de train est refusé s'il contient le split test.

In [ ]:
TASK = "smoke"  # "smoke", "train" ou "rebuild"
BRANCH = "codex/cleanup-cloud-training-docs"
REPO_URL = "https://github.com/Andriamarosoa/midi.git"
WORKSPACE = "/kaggle/working/midi"
WORKERS = 4


In [ ]:
import pathlib
import shutil
import subprocess
import sys

if not ((3, 9) <= sys.version_info[:2] < (3, 12)):
    raise RuntimeError(f"Python incompatible: {sys.version.split()[0]}")
workspace = pathlib.Path(WORKSPACE)
if workspace.exists():
    shutil.rmtree(workspace)
subprocess.run([
    "git", "clone", "--branch", BRANCH, "--single-branch",
    REPO_URL, str(workspace),
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--disable-pip-version-check", "-r",
    str(workspace / "requirements/cloud-training.txt"),
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-e",
    str(workspace), "--no-deps",
], check=True)


In [ ]:
command = [
    sys.executable,
    str(workspace / "scripts/cloud/kaggle_entrypoint.py"),
    "--task", TASK,
    "--input-root", "/kaggle/input",
    "--workers", str(WORKERS),
]
subprocess.run(command, cwd=workspace, check=True)


In [ ]:
for path in sorted((workspace / "runs").glob("**/cloud_pipeline.json")):
    print(path)
for path in sorted((workspace / "data/processed").glob("**/rebuild_report.json")):
    print(path)
